# Cross-check: Butler visitTable vs constDb visitTable

**Purpose:**  
Verify the equivalence of visit numbers, tracts and patches between two independent Rubin/LSST data sources:
- `visitTable` from the **Butler main** repository  (from https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/1-correlate-dp2-with-fink-alerts/notebooks/2026-03-25_FindObservationsInButlerRegistryInTractPatch.ipynb)
- `visitTable` from the **consDb** (consolidated database) (from https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_DP2_ConstDB_Butler_LSSTCam_VisitsTractPatch.ipynb)

This notebook also provides utilities to map diaObject alerts (from Fink lightcurves) to their tract/patch,
and to generate the `dataQuery` string for a `bps submit` DRP pipeline job.

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Date:** 2026-03-28


## 0. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR = "data_fromlsst"
DATA_OUTDIR = "data_tolsst"

# Butler main visitTable (with tracts/patches)
FILE_BUTLER = os.path.join(DATA_DIR, "visitTable-2025041500138-2026031900284_N52726_WithTracts.parquet")

# consDb visitTable (with tracts/patches)
FILE_CONSDB = os.path.join(
    DATA_DIR, "constDbVisitTable-2025041500043-2026032500675_N85366_WithTracts.parquet"
)

os.makedirs(DATA_OUTDIR, exist_ok=True)

print(f"Butler file : {FILE_BUTLER}")
print(f"consDb file : {FILE_CONSDB}")
print(f"Butler data exists: {os.path.exists(FILE_BUTLER)}")
print(f"consDb data exists: {os.path.exists(FILE_CONSDB)}")
print(f"Output data dir : {DATA_OUTDIR}")
print(f"Output data dir exists: {os.path.exists(DATA_OUTDIR)}")

## 1. Load the two tables

In [ ]:
df_butler = pd.read_parquet(FILE_BUTLER)
df_consdb = pd.read_parquet(FILE_CONSDB)

print("=" * 60)
print(f"Butler visitTable  →  {df_butler.shape[0]:,} rows  ×  {df_butler.shape[1]} columns")
print(f"consDb visitTable  →  {df_consdb.shape[0]:,} rows  ×  {df_consdb.shape[1]} columns")

### 1.1 Schema inspection

In [ ]:
print("── Butler columns ──")
print(df_butler.dtypes)
print()
print("── consDb columns ──")
print(df_consdb.dtypes)

In [ ]:
print("── Butler head ──")
display(df_butler.head(3))
print("── consDb head ──")
display(df_consdb.head(3))

### 1.2 Identify the key columns

Adjust these names if the actual column names in the parquet differ.

In [ ]:
# ── COLUMN NAME CONFIGURATION ─────────────────────────────────────────────────
# Adapt to the real column names found above.

# Visit identifier (13-digit Butler visitId or equivalent)
# COL_VISIT_BUTLER = "visitId"     # e.g. 'visitId', 'visit', 'r:visit'
COL_VISIT_BUTLER = "id"  # e.g. 'visitId', 'visit', 'r:visit'
COL_VISIT_CONSDB = "visit_id"  # e.g. 'visit_id', 'visitId'

# Sky-map geometry columns
COL_TRACT_BUTLER = "tract"
COL_PATCH_BUTLER = "patch"
COL_TRACT_CONSDB = "tract"
COL_PATCH_CONSDB = "patch"

# Optional: detector, band/filter
COL_DETECTOR_BUTLER = "detector"  # set to None if absent
COL_DETECTOR_CONSDB = "detector"
COL_BAND_BUTLER = "band"  # set to None if absent
COL_BAND_CONSDB = "band"


# Auto-detect: override config above with whatever is actually in the DataFrames
def detect_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            print(f"  [{label}] found column: '{c}'")
            return c
    print(f"  [{label}] WARNING – none of {candidates} found in columns")
    return None


print("Auto-detecting visit column …")
COL_VISIT_BUTLER = detect_col(df_butler, ["visitId", "visit", "r:visit", "visit_id", "id"], "butler")
COL_VISIT_CONSDB = detect_col(df_consdb, ["visit_id", "visitId", "visit", "r:visit", "id"], "consdb")

print("Auto-detecting tract/patch …")
COL_TRACT_BUTLER = detect_col(df_butler, ["tract", "skymap_tract"], "butler")
COL_PATCH_BUTLER = detect_col(df_butler, ["patch", "skymap_patch"], "butler")
COL_TRACT_CONSDB = detect_col(df_consdb, ["tract", "skymap_tract"], "consdb")
COL_PATCH_CONSDB = detect_col(df_consdb, ["patch", "skymap_patch"], "consdb")

print("Auto-detecting band …")
COL_BAND_BUTLER = detect_col(df_butler, ["band", "filter", "physical_filter"], "butler")
COL_BAND_CONSDB = detect_col(df_consdb, ["band", "filter", "physical_filter"], "consdb")

## 2. Visit-set comparison

In [ ]:
visits_butler = set(df_butler[COL_VISIT_BUTLER].dropna().astype(int))
visits_consdb = set(df_consdb[COL_VISIT_CONSDB].dropna().astype(int))

common = visits_butler & visits_consdb
only_butler = visits_butler - visits_consdb
only_consdb = visits_consdb - visits_butler

print(f"Visits in Butler only   : {len(visits_butler):,}")
print(f"Visits in consDb only   : {len(visits_consdb):,}")
print(f"Visits in BOTH          : {len(common):,}")
print(f"Only in Butler (not consDb): {len(only_butler):,}")
print(f"Only in consDb (not Butler): {len(only_consdb):,}")
print()
overlap_pct = 100 * len(common) / max(len(visits_butler), len(visits_consdb))
print(f"Overlap fraction (vs larger set): {overlap_pct:.1f} %")

In [ ]:
# Venn-style bar chart
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Butler only", "Common", "consDb only"]
values = [len(only_butler), len(common), len(only_consdb)]
colors = ["#4C9BE8", "#2ECC71", "#E8754C"]
bars = ax.bar(labels, values, color=colors, edgecolor="white", linewidth=1.4)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{val:,}",
        ha="center",
        va="bottom",
        fontsize=11,
    )
ax.set_ylabel("Number of visits")
ax.set_title("Visit overlap: Butler visitTable vs consDb visitTable")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 3. Tract / Patch consistency on common visits

For visits present in both tables, compare the (tract, patch) assignments.

In [ ]:
# Build merged table on common visits
cols_butler = [COL_VISIT_BUTLER, COL_TRACT_BUTLER, COL_PATCH_BUTLER]
cols_consdb = [COL_VISIT_CONSDB, COL_TRACT_CONSDB, COL_PATCH_CONSDB]
if COL_BAND_BUTLER:
    cols_butler.append(COL_BAND_BUTLER)
if COL_BAND_CONSDB:
    cols_consdb.append(COL_BAND_CONSDB)

sub_b = (
    df_butler[cols_butler]
    .copy()
    .rename(
        columns={
            COL_VISIT_BUTLER: "visitId",
            COL_TRACT_BUTLER: "tract_butler",
            COL_PATCH_BUTLER: "patch_butler",
        }
    )
)
if COL_BAND_BUTLER:
    sub_b = sub_b.rename(columns={COL_BAND_BUTLER: "band_butler"})

sub_c = (
    df_consdb[cols_consdb]
    .copy()
    .rename(
        columns={
            COL_VISIT_CONSDB: "visitId",
            COL_TRACT_CONSDB: "tract_consdb",
            COL_PATCH_CONSDB: "patch_consdb",
        }
    )
)
if COL_BAND_CONSDB:
    sub_c = sub_c.rename(columns={COL_BAND_CONSDB: "band_consdb"})

sub_b["visitId"] = sub_b["visitId"].astype(int)
sub_c["visitId"] = sub_c["visitId"].astype(int)

merged = sub_b.merge(sub_c, on="visitId", how="inner")
print(f"Merged table shape: {merged.shape}")
display(merged.head(5))

In [ ]:
# Tract agreement
merged["tract_match"] = merged["tract_butler"] == merged["tract_consdb"]
merged["patch_match"] = merged["patch_butler"].astype(str) == merged["patch_consdb"].astype(str)
merged["full_match"] = merged["tract_match"] & merged["patch_match"]

n_total = len(merged)
n_tract_ok = merged["tract_match"].sum()
n_patch_ok = merged["patch_match"].sum()
n_full_ok = merged["full_match"].sum()

print(f"Common visits (rows): {n_total:,}")
print(f"Tract matches  : {n_tract_ok:,} / {n_total:,}  ({100 * n_tract_ok / n_total:.2f} %)")
print(f"Patch matches  : {n_patch_ok:,} / {n_total:,}  ({100 * n_patch_ok / n_total:.2f} %)")
print(f"Full match     : {n_full_ok:,} / {n_total:,}  ({100 * n_full_ok / n_total:.2f} %)")

In [ ]:
# Show mismatches if any
mismatches = merged[~merged["full_match"]]
print(f"Mismatched rows: {len(mismatches)}")
if len(mismatches) > 0:
    display(mismatches.head(20))

### 3.1 Distribution of tracts and patches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tract distribution
ax = axes[0]
merged["tract_butler"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C9BE8")
ax.set_title("Tract distribution (Butler, common visits)")
ax.set_xlabel("tract")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)

# Patch distribution
ax = axes[1]
merged["patch_butler"].astype(str).value_counts().sort_index().plot(kind="bar", ax=ax, color="#E8754C")
ax.set_title("Patch distribution (Butler, common visits)")
ax.set_xlabel("patch")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 2D histogram: tract vs patch
tract_vals = merged["tract_butler"].astype(int)
patch_vals = merged["patch_butler"].astype(str)  # patch may be '0,0' style string

# If patch is numeric, plot 2D
try:
    patch_num = merged["patch_butler"].astype(int)
    fig, ax = plt.subplots(figsize=(9, 6))
    h = ax.hist2d(tract_vals, patch_num, bins=50, norm=LogNorm(), cmap="plasma")
    plt.colorbar(h[3], ax=ax, label="count (log scale)")
    ax.set_xlabel("tract")
    ax.set_ylabel("patch")
    ax.set_title("2D distribution: tract vs patch (Butler, common visits)")
    plt.tight_layout()
    plt.show()
except (ValueError, TypeError):
    print("Patch column is not purely numeric – skipping 2D histogram.")
    print("Unique patch values sample:", sorted(set(patch_vals.tolist()))[:20])

## 4. Build a canonical lookup table: visitId → (tract, patch)

This will be used later to map diaObject alerts to their sky-map position.

In [ ]:
# Prefer Butler as reference; fallback to consDb for visits absent from Butler
lookup_butler = sub_b.rename(columns={"tract_butler": "tract", "patch_butler": "patch"})
lookup_consdb = sub_c.rename(columns={"tract_consdb": "tract", "patch_consdb": "patch"})

lookup_all = pd.concat([lookup_butler, lookup_consdb], ignore_index=True)
lookup_all = lookup_all.drop_duplicates(subset=["visitId"]).set_index("visitId")

print(f"Canonical lookup table: {len(lookup_all):,} unique visitIds")
display(lookup_all.head(5))

## 5. Map diaObject alerts → tract / patch

This section loads the visit summary file produced by `01_fink_block_lightcurves.ipynb`
and cross-matches each alert's visitId against the canonical lookup table.

### Available alert files in `data_FINK_BLOCK_LC_01/`

| File | Format | Visit col | diaObjectId col | Notes |
|------|--------|-----------|-----------------|-------|
| `visit_summary_src.csv` | CSV | `visit` | `diaObjectId` | diaSources (detections) |
| `visit_summary_fp.csv`  | CSV | `visit` | `diaObjectId` | forced-photometry |
| `visit_index.csv`       | CSV | `r:visit` | `diaObjectId` | raw Fink index, prefixed columns |
| `visit_index_fp.csv`    | CSV | `r:visit` | `diaObjectId` | raw Fink index (fp) |
| `*_src.parquet` / `*_fp.parquet` | Parquet | `r:visit` | `i:diaObjectId` | per-class Fink alert data |

In [ ]:
# Inspect what is available in the Fink alert data folder
FINK_DATA_DIR = "../03_fink_api_blockselections/data_FINK_BLOCK_LC_01"
print(f"Files in {FINK_DATA_DIR}:")
for f in sorted(os.listdir(FINK_DATA_DIR)):
    fpath = os.path.join(FINK_DATA_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:<50s}  {size_kb:8.1f} KB")

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Choose a diaObject ID to investigate.
# Set DIAOBJECT_ID = None to use ALL diaObjects in the file.
DIAOBJECT_ID = 313853517449658448  # ← set this (or None for all)

# Path to the visit summary file generated by 01_fink_block_lightcurves.ipynb.
# Recommended: visit_summary_src.csv (diaSources) or visit_summary_fp.csv (forced phot).
# Both are CSVs with columns: diaObjectId, visit, detector, band, ...
ALERT_FILE = os.path.join(FINK_DATA_DIR, "visit_summary_src.csv")
# ──────────────────────────────────────────────────────────────────────────────

print(f"DIAOBJECT_ID : {DIAOBJECT_ID}")
print(f"ALERT_FILE   : {ALERT_FILE}")
print(f"File exists  : {os.path.exists(ALERT_FILE)}")

In [ ]:
def load_alert_file(path):
    """Load a visit summary/index file, auto-detecting CSV vs Parquet format.

    Normalises column names so the returned DataFrame always has:
      - 'visitId'     : int64 Butler visit number
      - 'diaObjectId' : int64
      - 'detector'    : int (if present)
      - 'band'        : str (if present)
    """
    ext = os.path.splitext(path)[1].lower()

    # ── 1. Read raw file ──────────────────────────────────────────────────────
    if ext == ".parquet":
        df = pd.read_parquet(path)
        fmt = "parquet"
    elif ext in (".csv", ".tsv"):
        df = pd.read_csv(path)
        fmt = "csv"
    else:
        # Sniff: try parquet first, fall back to csv
        try:
            df = pd.read_parquet(path)
            fmt = "parquet (sniffed)"
        except Exception:
            df = pd.read_csv(path)
            fmt = "csv (sniffed)"

    print(f"Loaded {fmt} file: {path}")
    print(f"  Raw shape   : {df.shape}")
    print(f"  Raw columns : {df.columns.tolist()}")

    # ── 2. Normalise visit column ─────────────────────────────────────────────
    # visit_summary_*.csv  →  'visit'
    # visit_index*.csv     →  'r:visit'
    # Fink parquet files   →  'r:visit'
    visit_candidates = ["visit", "r:visit", "visitId", "visit_id", "id"]
    visit_col = next((c for c in visit_candidates if c in df.columns), None)
    if visit_col is None:
        raise ValueError(f"Cannot find visit column in {df.columns.tolist()}")
    if visit_col != "visitId":
        df = df.rename(columns={visit_col: "visitId"})
    df["visitId"] = df["visitId"].astype(int)
    print(f"  Visit column: '{visit_col}' → 'visitId'")

    # ── 3. Normalise diaObjectId column ──────────────────────────────────────
    diaobj_candidates = ["diaObjectId", "i:diaObjectId", "objectId"]
    diaobj_col = next((c for c in diaobj_candidates if c in df.columns), None)
    if diaobj_col and diaobj_col != "diaObjectId":
        df = df.rename(columns={diaobj_col: "diaObjectId"})
        print(f"  diaObjectId column: '{diaobj_col}' → 'diaObjectId'")

    # ── 4. Normalise detector column ──────────────────────────────────────────
    det_candidates = ["detector", "r:detector"]
    det_col = next((c for c in det_candidates if c in df.columns), None)
    if det_col and det_col != "detector":
        df = df.rename(columns={det_col: "detector"})

    # ── 5. Normalise band column ───────────────────────────────────────────────
    band_candidates = ["band", "r:band", "i:fid", "filter"]
    band_col = next((c for c in band_candidates if c in df.columns), None)
    if band_col and band_col != "band":
        df = df.rename(columns={band_col: "band"})

    print(f"  Final shape : {df.shape}")
    return df


if ALERT_FILE is not None and os.path.exists(str(ALERT_FILE)):
    df_alerts_all = load_alert_file(ALERT_FILE)
    display(df_alerts_all.head(5))
else:
    raise FileNotFoundError(f"ALERT_FILE not found: {ALERT_FILE}")

In [ ]:
# Filter to the chosen diaObject (or keep all if DIAOBJECT_ID is None)
if DIAOBJECT_ID is not None and "diaObjectId" in df_alerts_all.columns:
    df_alerts = df_alerts_all[df_alerts_all["diaObjectId"] == DIAOBJECT_ID].copy()
    print(f"diaObjectId {DIAOBJECT_ID}: {len(df_alerts)} rows after filtering")
    if len(df_alerts) == 0:
        print()
        print("⚠️  No rows found for this diaObjectId. Available IDs:")
        print(df_alerts_all["diaObjectId"].unique()[:20])
else:
    df_alerts = df_alerts_all.copy()
    print(f"Using all diaObjects: {len(df_alerts)} rows")

display(df_alerts.head(5))

In [ ]:
# Cross-match alerts to canonical lookup table
df_matched = df_alerts.join(lookup_all[["tract", "patch"]], on="visitId", how="left")

n_matched = df_matched["tract"].notna().sum()
n_unmatched = df_matched["tract"].isna().sum()
print(f"Alerts matched to tract/patch    : {n_matched} / {len(df_alerts)}")
print(f"Unmatched (visitId not in lookup): {n_unmatched}")
if n_unmatched > 0:
    missing_visits = df_matched[df_matched["tract"].isna()]["visitId"].unique()
    print(f"Missing visitIds: {missing_visits[:10]}")
display(df_matched.head(10))

In [ ]:
# Check that all alerts belong to the SAME tract and patch (as expected for a point source)
unique_tracts = df_matched["tract"].dropna().unique()
unique_patches = df_matched["patch"].dropna().astype(str).unique()

print(f"Unique tracts  : {unique_tracts}")
print(f"Unique patches : {unique_patches}")

if len(unique_tracts) == 1 and len(unique_patches) == 1:
    print()
    print("✅  All alerts belong to a SINGLE tract/patch — consistent sky position.")
    TRACT = int(unique_tracts[0])
    PATCH = str(unique_patches[0])
    print(f"   tract = {TRACT}")
    print(f"   patch = {PATCH}")
elif len(unique_tracts) > 0:
    print()
    print("⚠️  Alerts span multiple tracts or patches — investigate further.")
    print(f"   tracts  : {unique_tracts}")
    print(f"   patches : {unique_patches}")
    TRACT, PATCH = None, None
else:
    print()
    print("❌  No tract/patch found — check visitId format or lookup table.")
    TRACT, PATCH = None, None

### histogram

In [ ]:
# Nettoyage minimal
df = df_matched.copy()
df["patch"] = df["patch"].astype(str)

# Comptage du nombre de visits uniques par (tract, patch)
df_count = df.groupby(["tract", "patch"])["visitId"].nunique().reset_index(name="n_visits")
df_count = df_count.sort_values(["tract", "n_visits"], ascending=[True, False])
df_count["label"] = "tract " + df_count["tract"].astype(str) + " | patch " + df_count["patch"]

In [ ]:
# Assigner une couleur par tract
tracts = df_count["tract"].unique()


tracts = df_count["tract"].unique()
cmap = plt.get_cmap("tab10")

# mapping correct → couleurs RGBA
tract_to_color = {t: cmap(i % cmap.N) for i, t in enumerate(tracts)}

colors = df_count["tract"].map(tract_to_color)
# fallback si NaN
colors = colors.fillna("gray")

plt.figure(figsize=(8, 6))

plt.barh(df_count["label"], df_count["n_visits"], color=colors)

plt.xlabel("Number of visits")
plt.ylabel("Tract / Patch")
plt.title("Visits per tract/patch (color-coded by tract)")

plt.tight_layout()
plt.show()

## 6. Save visit file

### 6.1 Find which tract win the count majority

In [ ]:
df = df_matched.copy()

tract_counts = df.groupby("tract")["visitId"].nunique().sort_values(ascending=False)

print(tract_counts)
TRACT = int(tract_counts.index[0])
print(f"Tract majoritaire = {TRACT}")

### 6.2 Select the visits in that tract

In [ ]:
df_sel = df[df["tract"] == TRACT]

### 6.3 Extract the visit list and the DIAOBJ_ID

In [ ]:
visit_list = sorted(df_sel["visitId"].dropna().astype(int).unique())
DIAOBJECT_ID = int(df_sel["diaObjectId"].iloc[0])

### 6.4 Write the visit file

In [ ]:
filename = f"{DATA_OUTDIR}/diaObj{DIAOBJECT_ID}_tract{TRACT}.txt"

with open(filename, "w") as f:
    for v in visit_list:
        f.write(f"{v}\n")

print(f"Fichier écrit : {filename}")
print(f"Nombre de visits : {len(visit_list)}")

## 7. Generate the `dataQuery` string for `bps submit`

The DRP pipeline is launched via `bps submit <yaml>` with a `dataQuery` that selects the relevant
visits and sky-map position.

- see my example to run the DRP myself here : https://github.com/sylvielsstfr/MyRubinLSSTpipelines/blob/main/01_LSSTCamPhotometry/run_bps_pipeline.sh
  
- see my example to inspect official drp-dp2prep here :


  - viewing depp_coadds with matplotlib : https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_LSSTCamDeepCoaddsMosaicMatplotlib.ipynb
    
  - viewing depp_coadds with firefly : https://github.com/sylvielsstfr/RubinLSSTDP2Data/blob/main/notebooks/2026-03-26_LSSTCamDeepCoaddsMosaicFirefly.ipynb
 

#### My Butler/ DP2 actual collections in the butler `dp2_prep`
```python
# The output repo is tagged with the Jira ticket number "DM-40356":
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
where_clause = "instrument = '" + instrument + "'"
```

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
SKYMAP = "lsst_cells_v2"  # adjust to actual skymap used at USDF
INSTRUMENT = "LSSTCam"  # or LSSTComCam / HSC for tests
COLLECTIONS = "LSSTCam/Defaults"  # ← update
# ──────────────────────────────────────────────────────────────────────────────

if TRACT is not None:
    # All visitIds for this diaObject
    visit_list = sorted(df_matched["visitId"].dropna().astype(int).unique().tolist())
    visit_str = ", ".join(str(v) for v in visit_list)

    data_query = (
        f"instrument='{INSTRUMENT}' AND skymap='{SKYMAP}' AND tract={TRACT} AND visit IN ({visit_str})"
    )

    print("=" * 70)
    print("dataQuery for bps submit:")
    print("=" * 70)
    print(data_query)
    print("=" * 70)
    print()
    print(f"Number of visits in query : {len(visit_list)}")
    print(f"Visit range               : {visit_list[0]} … {visit_list[-1]}")

    # Optional: write to a text file for copy-paste into the BPS yaml
    query_file = f"{DATA_OUTDIR}/dataQuery_diaObj{DIAOBJECT_ID}_tract{TRACT}_patch{PATCH}.txt"
    with open(query_file, "w") as fh:
        fh.write(data_query + "\n")
    print(f"\nQuery saved to: {query_file}")
else:
    print("Cannot generate dataQuery: tract/patch not uniquely determined.")

### 6.1 BPS YAML snippet

In [ ]:
if TRACT is not None:
    yaml_snippet = f"""
# ── BPS submit YAML snippet ────────────────────────────────────────────
project: drp_diaObject_{DIAOBJECT_ID}
campaign: tract{TRACT}

pipelineYaml: "$DRP_PIPE_DIR/pipelines/LSSTCam/DRP.yaml"

payload:
  butlerConfig: /repo/main
  inCollection: {COLLECTIONS}
  dataQuery: >
    {data_query}
# ──────────────────────────────────────────────────────────────────────
"""
    print(yaml_snippet)

## 7. Summary

| Item | Value |
|------|-------|
| Butler visits | — |
| consDb visits | — |
| Common visits | — |
| Full tract+patch match | — |
| diaObject tract | — |
| diaObject patch | — |

*Fill in from the cell outputs above.*